# Cloning Repo



In [1]:
import os
import subprocess
from urllib.parse import urlparse

def repo_clone(repo_url):
    parsed = urlparse(repo_url)

    if parsed.scheme != "https" or parsed.hostname != "github.com":
        raise ValueError("Please provide a valid HTTPS GitHub URL.")

    parts = parsed.path.strip("/").split("/")
    if len(parts) != 2 or not all(parts):
        raise ValueError("Expected a URL like https://github.com/owner/repo")

    repo_name = parts[1].removesuffix(".git")
    target_path = os.path.join("/kaggle/working", repo_name)

    if not os.path.isdir(os.path.join(target_path, ".git")):
        if os.path.exists(target_path):
            raise ValueError("Target directory exists but is not a Git repository.")

        subprocess.run(
            ["git", "clone", "--depth", "1", repo_url, target_path],
            check=True,
            capture_output=True,
            text=True
        )

    return target_path

In [2]:
def read_source_files(repo):
    # Define common source file extensions we want to read
    source_extensions = (
        '.py', '.js', '.jsx', '.ts', '.tsx', '.html', '.css',
        '.java', '.cpp', '.c', '.h', '.cs', '.go', '.rs', '.kt',
        '.swift', '.sh', '.md', '.json', '.rb', '.erb'
    )

    # Directories to completely ignore
    exclude_dirs = {'.git', '__pycache__', 'node_modules', '.ipynb_checkpoints', 'venv', 'env'}

    source_contents = {}

    if not os.path.exists(repo):
        print(f"Directory '{repo}' does not exist.")
        return source_contents

    print(f"Scanning and reading source files in: {repo}\n")
    for root, dirs, files in os.walk(repo):
        # Modifying dirs in-place to skip excluded directories
        dirs[:] = [d for d in dirs if d not in exclude_dirs]

        for file in files:
            if file.endswith(source_extensions):
                file_path = os.path.join(root, file)
                relative_path = os.path.relpath(file_path, repo)
                try:
                    with open(file_path, 'r', encoding='utf-8') as f:
                        content = f.read()
                        source_contents[relative_path] = content
                        print(f"Loaded: {relative_path} ({len(content)} characters)")
                except Exception as e:
                    print(f"Could not read {relative_path}: {e}")

    return source_contents

# Read source files from the cloned repository
def read_files_validation(repo_name):
    if 'repo_name' in globals() and os.path.exists(repo_name):
        source_files = read_source_files(repo_name)
        print(f"\nSuccessfully read {len(source_files)} source files.")
    else:
        print("Repository path not found. Please ensure the repository is cloned properly first.")
    return

# Huggingface token


In [3]:
from huggingface_hub import login
login()

# Mistral model

In [4]:
!pip install transformers==4.52.4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 95.1 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 85.7 MB/s eta 0:00:00:00:01
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.29.0
    Uninstalling huggingface_hub-1.29.0:
      Successfully uninstalled huggingface_hub-1.29.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.23.1
    Uninstalling tokenizers-0.23.1:
      Successfully uninstalled tokenizers-0.23.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.16.1
    Uninstalling transformers-5.16.1:
      Successfully uninstalled transformers-5.16.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingf

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
).eval()

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  493kB            

tokenizer.model: downloading bytes:           |  0.00B            

...

model.safetensors.index.json:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [5]:
def generate_text(prompt, max_new_tokens=512):
    messages = [{"role": "user", "content": prompt}]

    inputs = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        return_dict=True,
        add_generation_prompt=True,
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.4,
        pad_token_id=tokenizer.eos_token_id,
    )

    input_length = inputs["input_ids"].shape[1]
    generated = outputs[0][input_length:]

    return tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

# RAG with LangChain

In [6]:
!pip install langchain-community faiss-cpu langchain_huggingface langchain_text_splitters torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 25.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 89.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.0/573.0 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.0 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.6.1
    Uninstalling langchain-core-1.6.1:
      Successfully uninstalled langchain-core-1.6.1


In [7]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
# from langchain_text_splitters import CharacterTextSplitter
import torch

/tmp/ipykernel_48/545512326.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [8]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
)

def rag(files):
    texts, metadatas = [], []
    for path, content in files.items():
        for chunk in splitter.split_text(content):
            texts.append(chunk)
            metadatas.append({"path": path})

    embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embedding = HuggingFaceEmbeddings(model_name=embedding_model_name)
    vectordb = FAISS.from_texts(texts, embedding, metadatas= metadatas)
    return vectordb

In [9]:
vectordb = None

def initialize_repository(repo_url):
    global vectordb

    repo_path = repo_clone(repo_url)
    files = read_source_files(repo_path)

    if not files:
        raise ValueError("No supported source files were found.")

    vectordb = rag(files)
    return len(files)
    

In [10]:
def answer_question(question):
    if vectordb is None:
        raise RuntimeError("Load a repository before asking questions.")
    docs = vectordb.similarity_search(question, k=3)

    context = "\n\n".join(
        doc.page_content for doc in docs
    )

    prompt = f"""You are a codebase assistant.
Answer using the provided source code.
If the context is insufficient, say so.

Context:
{context}

Question: {question}

Answer:"""

    # format_instructions, code_template = response_format()
    # return generate_response(question, format_instructions, code_template).strip()
    return generate_text(prompt, max_new_tokens = 1200)

In [11]:

from fastapi import FastAPI, HTTPException, Request
from pydantic import BaseModel
import uvicorn, threading, time


app = FastAPI()

class RepoRequest(BaseModel):
    repo_url: str

class QuestionRequest(BaseModel):
    question: str

@app.post("/load_repo")
def api_load_repo(request: RepoRequest):
    try:
        count = initialize_repository(request.repo_url)
        return {
            "message": "Repository loaded successfully",
            "files_indexed": count
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/ask")
def api_ask(request: QuestionRequest):
    try:
        answer = answer_question(request.question)
        return {"answer": answer}
    except RuntimeError as e:
        raise HTTPException(status_code=400, detail=str(e))
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

In [12]:
!pip install pyngrok

In [13]:
import socket
from pyngrok import ngrok, conf
ngrokToken = "3KQXt4f3JfqvHuIESZdCqOERVDw_7zGGzFNmTqEmXjkP8jrVP"


def free_port():
    s = socket.socket()
    s.bind(('', 0))
    port = s.getsockname()[1]
    s.close()
    return port

port = free_port()
conf.get_default().auth_token = ngrokToken
public_url = ngrok.connect(port).public_url
print("Your public URL:", public_url)

def run(): uvicorn.run(app, host="0.0.0.0", port=port)
threading.Thread(target=run, daemon=True).start()
time.sleep(1)

INFO:     Started server process [48]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:49845 (Press CTRL+C to quit)


Your public URL: https://shakily-smooth-scrubbed.ngrok-free.dev
